# Chapter 06 — SOLUTIONS: Metrics & Evaluation

**Instructor file — share only after exercise session.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score,
    roc_curve, roc_auc_score
)

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Regression setup
diabetes = load_diabetes()
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42)
lr = Pipeline([('s', StandardScaler()), ('m', LinearRegression())])
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
lr.fit(X_reg_train, y_reg_train)
rf_reg.fit(X_reg_train, y_reg_train)
y_pred_lr  = lr.predict(X_reg_test)
y_pred_rf  = rf_reg.predict(X_reg_test)

# Classification setup
cancer = load_breast_cancer()
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target)
clf = Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))])
clf.fit(X_clf_train, y_clf_train)
y_clf_pred  = clf.predict(X_clf_test)
y_clf_proba = clf.predict_proba(X_clf_test)[:, 1]

print('Setup complete!')

## Part A Solution: Regression Metrics

In [ ]:
# A1: Compute metrics
for name, preds in [('Linear Regression', y_pred_lr), ('Random Forest', y_pred_rf)]:
    mae  = mean_absolute_error(y_reg_test, preds)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
    r2   = r2_score(y_reg_test, preds)
    print(f'{name:<22}  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}')

print('\n→ Both perform similarly here — Linear Regression is surprisingly competitive')
print('  because the diabetes dataset is roughly linear in these features.')

In [ ]:
# A2: Residual plot for better model
best_preds = y_pred_rf if r2_score(y_reg_test, y_pred_rf) > r2_score(y_reg_test, y_pred_lr) else y_pred_lr
best_name  = 'Random Forest' if best_preds is y_pred_rf else 'Linear Regression'
residuals = y_reg_test - best_preds

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(best_preds, residuals, alpha=0.4, s=25, color='#3498db')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Predicted Value')
ax.set_ylabel('Residual (Actual − Predicted)')
ax.set_title(f'{best_name} — Residual Plot', fontsize=13)
plt.tight_layout()
plt.show()
print('Residuals appear fairly random → no obvious systematic error.')

## Part B Solution: Classification Metrics

In [ ]:
# B1: Metrics
acc  = accuracy_score(y_clf_test, y_clf_pred)
prec = precision_score(y_clf_test, y_clf_pred)
rec  = recall_score(y_clf_test, y_clf_pred)
f1   = f1_score(y_clf_test, y_clf_pred)

print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}  ← MOST IMPORTANT for cancer detection')
print(f'F1 Score:  {f1:.4f}')
print()
print('For cancer detection, recall (sensitivity) is most critical.')
print('We must minimize False Negatives (missed cancer cases).')
print('A missed cancer is far more costly than a false alarm.')

In [ ]:
# B2: Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_clf_test, y_clf_pred)
ConfusionMatrixDisplay(cm, display_labels=['malignant', 'benign']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Cancer Detection', fontsize=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'False Negatives (missed cancer): {fn} → these are the dangerous errors!')
print(f'False Positives (false alarm):   {fp} → concerning but less dangerous')

In [ ]:
# B3: ROC Curve
fpr, tpr, _ = roc_curve(y_clf_test, y_clf_proba)
auc = roc_auc_score(y_clf_test, y_clf_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#3498db', linewidth=2.5, label=f'AUC = {auc:.3f}')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#3498db')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — Cancer Detection', fontsize=13)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()
print(f'AUC = {auc:.4f} → excellent! Much better than random guessing (0.5)')

In [ ]:
# B4 Bonus: Threshold adjustment
for threshold in [0.5, 0.3, 0.2]:
    y_custom = (y_clf_proba > threshold).astype(int)
    prec_t = precision_score(y_clf_test, y_custom)
    rec_t  = recall_score(y_clf_test, y_custom)
    fn_t   = confusion_matrix(y_clf_test, y_custom).ravel()[2]
    print(f'Threshold={threshold}:  Precision={prec_t:.3f}  Recall={rec_t:.3f}  FN={fn_t}')

print('\nLower threshold → higher recall (catch more cancer) → lower precision (more false alarms)')
print('In life-critical applications, we often prefer threshold < 0.5 to maximize recall.')